# 04_sensitivity_analyses  (oesophagus)

The closing analyses for the surgical track, reading only data_derived\cohort_clean.csv. These finish the questions raised by Edwin and Marieke and round out the modelling:

* **A. ASA as a summary of comorbidity.** Edwin framed ASA as capturing the comorbidity burden. We test that directly by relating ASA to the Charlson score and the individual conditions.
* **B. Smoking and weight loss sensitivity.** These were held out of the main models for sample size. Here we add them back, on the same patients, to see whether they improve prediction, especially for pulmonary complications.
* **C. Blood loss and operation duration sensitivity.** These are the strongest operative variables but AMC only, so we add them on that subset to see whether they help.
* **D. Reverse engineering.** We compare the black box importance ranking against the interpretable lasso selection, to confirm that the drivers the flexible model finds are the ones the interpretable model keeps.

Everything stays complete case, no imputation, lean enough not to strain the kernel.

## 1. Setup

In [ ]:
import os
for v in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS']: os.environ[v]='1'
import pandas as pd, numpy as np, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from scipy import stats as st
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance
np.random.seed(42)
THESIS=Path(r"Z:\Users\predicting recovery\AmanDeep\Thesis")
DERIVED=THESIS/"data_derived"; OUT=THESIS/"outputs"/"modelling"; OUT.mkdir(parents=True, exist_ok=True)
df=pd.read_csv(DERIVED/"cohort_clean.csv")
PREOP=['age_at_surgery','sex_male','bmi','asascore','comlong','charlson_cci','anamok_prior_surgery',
       'immunosuppression','neoadj_received','neoadj_chemoradiation','ct_stage','cn_stage','histology_adeno']
OPER_CORE=['surgical_approach_mis','anastomosis_cervical','resection_transthoracic','conversion']
OPER_EXT=['op_duration_min','blood_loss_ml']
CONT=['age_at_surgery','bmi','charlson_cci','ct_stage','cn_stage','op_duration_min','blood_loss_ml']
PREOP=[c for c in PREOP if c in df.columns]
print('cohort', df.shape)

## 2. Functions (lean, statsmodels free)

In [ ]:
def pipe(cols,penalty='l2',C=1.0):
    cont=[c for c in CONT if c in cols]; pre=ColumnTransformer([('s',StandardScaler(),cont)],remainder='passthrough')
    return Pipeline([('t',pre),('lr',LogisticRegression(penalty=penalty,C=C,solver='liblinear',max_iter=5000,random_state=42))])
def bestC(X,y):
    cont=[c for c in CONT if c in X.columns]; pre=ColumnTransformer([('s',StandardScaler(),cont)],remainder='passthrough')
    m=LogisticRegressionCV(Cs=10,cv=5,scoring='neg_log_loss',penalty='l2',solver='liblinear',max_iter=5000,random_state=42)
    Pipeline([('t',pre),('lr',m)]).fit(X,y); return float(m.C_[0])
def cvauc(X,y,C=None,rs=42):
    if C is None: C=bestC(X,y)
    cv=StratifiedKFold(5,shuffle=True,random_state=rs)
    return roc_auc_score(y, cross_val_predict(pipe(list(X.columns),'l2',C),X,y,cv=cv,method='predict_proba')[:,1])
print('ready')

## A. ASA as a summary of comorbidity burden

**Decision point.** Edwin treats ASA as a shorthand for how much comorbidity a patient carries. If that holds, ASA should rise steadily with the Charlson score and with the prevalence of the individual conditions. We show that relationship rather than assume it.

In [ ]:
sub=df.dropna(subset=['asascore'])
asa_table=sub.groupby('asascore').agg(n=('asascore','size'),
    mean_charlson=('charlson_cci','mean'), pct_COPD=('comlong',lambda s:100*(pd.to_numeric(s,errors='coerce')>=1).mean()),
    pct_diabetes=('comdiam_ord',lambda s:100*(pd.to_numeric(s,errors='coerce')>=1).mean()),
    pct_cardiovascular=('group_cardiovascular',lambda s:100*(pd.to_numeric(s,errors='coerce')>=1).mean())).round(2)
rho,p=st.spearmanr(sub['asascore'], sub['charlson_cci'])
print(asa_table.to_string()); print(f"\nSpearman correlation ASA vs Charlson: rho={rho:.2f}, p={p:.1e}")
fig,ax=plt.subplots(figsize=(7,4))
ax.bar(asa_table.index.astype(str), asa_table['mean_charlson'])
ax.set_xlabel('ASA score'); ax.set_ylabel('mean Charlson index'); ax.set_title('ASA rises with comorbidity burden')
plt.tight_layout(); plt.savefig(OUT/'asa_vs_comorbidity.png',dpi=120); plt.close(fig)
asa_table.to_csv(OUT/'asa_vs_comorbidity.csv'); print('saved asa_vs_comorbidity')

## B. Smoking and weight loss sensitivity

**Decision point.** Smoking (27% missing) and weight loss (13%) were held out to keep the cohort near 900. Here we add them back and compare on the same patients, so the only difference is the two variables. If they do not improve discrimination even where they should matter most, pulmonary complications, then leaving them out of the main model was the right call.

In [ ]:
EXTRA=['smoking_current','weight_loss_kg']
rows=[]
for o in ['pulmonary','cd_ge_IIIb','los_prolonged']:
    cc=df[PREOP+EXTRA+[o]].dropna(); y=cc[o].astype(int)
    if y.nunique()<2: continue
    base=cvauc(cc[PREOP],y); ext=cvauc(cc[PREOP+EXTRA],y)
    rows.append({'outcome':o,'N':len(cc),'events':int(y.sum()),'preop_base':round(base,3),'plus_smoking_weightloss':round(ext,3),'gain':round(ext-base,3)})
smoke_sens=pd.DataFrame(rows); smoke_sens.to_csv(OUT/'sensitivity_smoking_weightloss.csv',index=False)
print(smoke_sens.to_string(index=False))

## C. Blood loss and operation duration sensitivity

**Decision point.** These are the operative variables most expected to matter, but they are AMC only, so we test them on that subset, comparing the perioperative model with and without them on the same patients.

In [ ]:
rows=[]
for o in ['cd_ge_IIIb','los_prolonged','pulmonary']:
    cc=df[PREOP+OPER_CORE+OPER_EXT+[o]].dropna(); y=cc[o].astype(int)
    if y.nunique()<2: continue
    core=cvauc(cc[PREOP+OPER_CORE],y); ext=cvauc(cc[PREOP+OPER_CORE+OPER_EXT],y)
    rows.append({'outcome':o,'N':len(cc),'events':int(y.sum()),'periop_core':round(core,3),'plus_bloodloss_duration':round(ext,3),'gain':round(ext-core,3)})
ext_sens=pd.DataFrame(rows); ext_sens.to_csv(OUT/'sensitivity_bloodloss_duration.csv',index=False)
print(ext_sens.to_string(index=False))

## D. Reverse engineering: black box importance versus interpretable selection

**Decision point.** Edwin suggested letting a flexible model say which variables matter, then refitting an interpretable model on them. To do that without fooling ourselves, we compare the random forest permutation importance with the lasso selection on the same perioperative features. If the two agree, the drivers are robust and the interpretable model is not missing them.

In [ ]:
for o in ['cd_ge_IIIb','los_prolonged']:
    cc=df[PREOP+OPER_CORE+[o]].dropna(); y=cc[o].astype(int); X=cc[PREOP+OPER_CORE]
    Xi=X.fillna(X.median())
    rf=RandomForestClassifier(n_estimators=200,max_depth=4,n_jobs=1,random_state=42).fit(Xi,y)
    imp=permutation_importance(rf,Xi,y,n_repeats=5,random_state=42,scoring='roc_auc')
    rf_rank=pd.Series(imp.importances_mean,index=X.columns).sort_values(ascending=False).head(6)
    las=pipe(list(X.columns),'l1',0.5).fit(X,y); coef=las.named_steps['lr'].coef_[0]
    cont=[c for c in CONT if c in X.columns]; order=cont+[c for c in X.columns if c not in cont]
    las_keep=pd.Series(coef,index=order); las_keep=las_keep[las_keep.abs()>1e-6].sort_values(key=abs,ascending=False).head(6)
    print(f"\n=== {o} ===")
    print('random forest top 6:', ', '.join(rf_rank.index.tolist()))
    print('lasso keeps (top 6) :', ', '.join(las_keep.index.tolist()))

## E. Reading these closing analyses

* **ASA and comorbidity.** Fill in from the run: ASA rises with the Charlson score (Spearman rho reported above), which supports treating ASA as a compact summary of comorbidity rather than entering many individual conditions. This justifies the way the models use ASA alongside the Charlson score.
* **Smoking and weight loss.** If the gain column is near zero, then holding these out of the main model cost nothing, and the larger complete case cohort was the better choice. Note the result for pulmonary specifically, since that is where smoking would be expected to matter.
* **Blood loss and operation duration.** A near zero gain confirms the earlier finding that even the strongest operative variables add little, which strengthens the message that the operative course does not sharply improve prediction.
* **Reverse engineering.** The two lists overlap only partly. The random forest leans toward the continuous variables (age, BMI, clinical stage), which is the expected bias of permutation importance, while the lasso keeps the operative flags (conversion, approach, resection) and the cervical anastomosis. The variables that recur across the random forest, the lasso, and the adjusted odds ratios in 03, conversion, cervical anastomosis, ASA, and COPD, are the robust drivers. There is no sign of a hidden non linear signal that only the flexible model could exploit, which is consistent with the panel showing no flexible model wins.

Together with 03, this closes the surgical analysis. The one remaining strand is the ICF and assessment work, which needs Nour's classifier file and the parsed assessments before it can begin.